In [1]:
!pip -q install -U "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects; we use bitsandbytes
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 96.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.5 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 

In [2]:
#Config (the single source of truth) + helpers
import os, json, re
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
DATA = "/kaggle/input/datasets/ianrusseladem/pipeline2-data"
OUT  = "/kaggle/working"

CFG = {
    "name": "helpdesk-resolution",
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "lora": {"rank": 16, "alpha": 32, "dropout": 0.05, "target_modules": "all-linear"},
    "train": {"epochs": 6, "lr": 2e-4, "max_len": 1408, "batch": 1, "grad_accum": 16},
    "guardrails": {"replay_mix": True},                 # train_mix vs train_synth
    "acceptance": {"min_task_gain_macro_f1": 0.05, "max_regression_drop": 0.05},
    "adjust_on_fail": {"force_replay": True, "lower_lr_factor": 0.5, "reduce_epochs_to": 2},
}
LABELS = [l.strip() for l in open(f"{DATA}/labels.txt") if l.strip()] or ["Done", "Won't Do"]

def read_jsonl(p): return [json.loads(l) for l in open(p) if l.strip()]
def normalize(s): return " ".join(s.lower().split())
def c_predict_label(out, labels):
    tail = out.rsplit("</think>", 1)[-1] if "</think>" in out else out
    by = {normalize(l): l for l in labels}
    lines = [x for x in tail.splitlines() if x.strip()]
    last = normalize(lines[-1]) if lines else ""
    if last in by: return by[last]
    low = normalize(tail); hits = [l for l in labels if normalize(l) in low]
    return max(hits, key=len) if hits else None
def macro_f1(gold, pred, labels):
    tot = 0.0
    for l in labels:
        tp = sum(1 for g,p in zip(gold,pred) if g==l and p==l)
        fp = sum(1 for g,p in zip(gold,pred) if g!=l and p==l)
        fn = sum(1 for g,p in zip(gold,pred) if g==l and p!=l)
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        tot += 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return tot/len(labels)
def score_probes(raw, probes, mode):
    ok = 0
    for o, p in zip(raw, probes):
        want = [a.lower() for a in p["answers"]]; low = o.lower()
        ok += (all(a in low for a in want) if mode=="all" else any(a in low for a in want))
    return ok/len(probes) if probes else 0.0
print("config ready; DATA:", os.listdir(DATA) if os.path.isdir(DATA) else "ADD THE DATASET")

config ready; DATA: ['train_synth.jsonl', 'train_mix.jsonl', 'sentinel.jsonl', 'labels.txt', 'gold.jsonl', 'reasoning_probes.jsonl', 'tool_probes.jsonl']


In [3]:
#Config-driven training
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def train_one(data_file, run_name, lr=None, epochs=None):
    t, lo = CFG["train"], CFG["lora"]
    lr = t["lr"] if lr is None else lr
    epochs = t["epochs"] if epochs is None else epochs
    tok = AutoTokenizer.from_pretrained(CFG["base_model"])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(CFG["base_model"], quantization_config=quant,
        torch_dtype=torch.bfloat16, device_map="auto"); model.config.use_cache = False
    lora = LoraConfig(r=lo["rank"], lora_alpha=lo["alpha"], lora_dropout=lo["dropout"],
        bias="none", task_type="CAUSAL_LM", target_modules=lo["target_modules"])
    ds = load_dataset("json", data_files=f"{DATA}/{data_file}", split="train")
    out = f"{OUT}/{run_name}"
    cfg = SFTConfig(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=t["batch"],
        gradient_accumulation_steps=t["grad_accum"], learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=10, save_strategy="epoch", bf16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=t["max_len"], packing=False, assistant_only_loss=True, seed=0, report_to="none")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok)
    r = tr.train(); tr.save_model(out); tok.save_pretrained(out)
    print(f"[train] {run_name}: loss={r.training_loss:.4f} (data={data_file}, lr={lr}, epochs={epochs})")
    return out

In [4]:
# Two-axis evaluation 
def _load(adapter=None):
    tok = AutoTokenizer.from_pretrained(CFG["base_model"]); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(CFG["base_model"], torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), tok

@torch.no_grad()
def _gen(m, tok, prompts, max_new, batch=16):
    outs = []
    for i in range(0, len(prompts), batch):
        ch = prompts[i:i+batch]
        txt = [tok.apply_chat_template(x, add_generation_prompt=True, tokenize=False) for x in ch]
        enc = tok(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(ch)):
            outs.append(tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

def _probe(m, tok, fname, mode, max_new):
    probes = read_jsonl(f"{DATA}/{fname}")
    raw = _gen(m, tok, [[{"role":"user","content":p["question"]}] for p in probes], max_new)
    return {"n": len(probes), "score": score_probes(raw, probes, mode)}

def evaluate(name, adapter=None):
    gold = read_jsonl(f"{DATA}/gold.jsonl")
    m, tok = _load(adapter)
    raw = _gen(m, tok, [r["messages"] for r in gold], 384)
    g = [r["label"] for r in gold]; p = [c_predict_label(o, LABELS) for o in raw]
    task = {"macro_f1": macro_f1(g, p, LABELS),
            "accuracy": sum(1 for a,b in zip(g,p) if a==b)/len(g),
            "valid": sum(1 for x in p if x is not None)/len(g)}
    res = {"name": name, "task": task,
           "sentinel": _probe(m, tok, "sentinel.jsonl", "any", 32),
           "reasoning": _probe(m, tok, "reasoning_probes.jsonl", "any", 256),
           "tools": _probe(m, tok, "tool_probes.jsonl", "all", 128)}
    json.dump(res, open(f"{OUT}/result_{name}.json","w"), indent=2)
    print(f"[eval] {name}: macroF1={task['macro_f1']:.3f} sentinel={res['sentinel']['score']:.3f} "
          f"reasoning={res['reasoning']['score']:.3f} tools={res['tools']['score']:.3f}")
    del m; torch.cuda.empty_cache(); return res

In [5]:
#The gate  + the orchestrated loop
def gate(base, cand):
    a = CFG["acceptance"]
    task_gain = cand["task"]["macro_f1"] - base["task"]["macro_f1"]
    worst = max((base[k]["score"] - cand[k]["score"]) for k in ["sentinel","reasoning","tools"])
    accept = task_gain >= a["min_task_gain_macro_f1"] and worst <= a["max_regression_drop"]
    print(f"[gate] task_gain={task_gain:+.3f} (min {a['min_task_gain_macro_f1']:+.3f}) | "
          f"worst_drop={worst:.3f} (max {a['max_regression_drop']:.3f}) -> "
          f"{'ACCEPT' if accept else 'REJECT'}")
    return accept, task_gain, worst

base = evaluate("base")
replay = CFG["guardrails"]["replay_mix"]
name1 = f"{CFG['name']}-{'replay' if replay else 'noreplay'}"
adapter1 = train_one("train_mix.jsonl" if replay else "train_synth.jsonl", name1)
cand1 = evaluate(name1, adapter1)
accepted, *_ = gate(base, cand1)
final = adapter1 if accepted else None

if not accepted:                       # one adjusted re-run
    adj = CFG["adjust_on_fail"]
    lr = CFG["train"]["lr"] * adj["lower_lr_factor"]
    print(f"[pipeline] gate rejected; adjusted re-run (replay on, lr={lr}, epochs={adj['reduce_epochs_to']})")
    adapter2 = train_one("train_mix.jsonl", f"{CFG['name']}-adj", lr=lr, epochs=adj["reduce_epochs_to"])
    cand2 = evaluate(f"{CFG['name']}-adj", adapter2)
    ok2, *_ = gate(base, cand2)
    final = adapter2 if ok2 else None
print("\nACCEPTED:", final or "none")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.218691
20,2.212719
30,1.876596
40,1.668847
50,1.511045
60,1.513128


[train] helpdesk-resolution-replay: loss=2.0002 (data=train_mix.jsonl, lr=0.0002, epochs=6)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[eval] helpdesk-resolution-replay: macroF1=0.501 sentinel=0.967 reasoning=0.933 tools=0.867
[gate] task_gain=+0.106 (min +0.050) | worst_drop=-0.066 (max 0.050) -> ACCEPT

ACCEPTED: /kaggle/working/helpdesk-resolution-replay


In [6]:
## (Optional) the guardrail experiment: no-replay vs replay

nr = train_one("train_synth.jsonl", f"{CFG['name']}-noreplay")
res_nr = evaluate(f"{CFG['name']}-noreplay", nr)
print("\nregression axis, base vs no-replay vs replay (replay should hold the probes up):")
for tag, r in [("base", base), ("no-replay", res_nr), ("replay", cand1)]:
    print(f"  {tag:<9} macroF1={r['task']['macro_f1']:.3f}  sentinel={r['sentinel']['score']:.3f}  "
          f"reasoning={r['reasoning']['score']:.3f}  tools={r['tools']['score']:.3f}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/tmp/ipykernel_108/1418843985.py:22: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  cfg = SFTConfig(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=t["batch"],


Tokenizing train dataset:   0%|          | 0/222 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.960445
20,2.697087
30,2.319221
40,2.176853


[train] helpdesk-resolution-noreplay: loss=2.7525 (data=train_synth.jsonl, lr=0.0002, epochs=6)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[eval] helpdesk-resolution-noreplay: macroF1=0.490 sentinel=0.967 reasoning=0.900 tools=0.900

regression axis, base vs no-replay vs replay (replay should hold the probes up):
  base      macroF1=0.395  sentinel=0.902  reasoning=0.850  tools=0.783
  no-replay macroF1=0.490  sentinel=0.967  reasoning=0.900  tools=0.900
  replay    macroF1=0.501  sentinel=0.967  reasoning=0.933  tools=0.867


In [7]:
##  Download adapters + results
import shutil
for n in [name1, f"{CFG['name']}-adj"]:
    if os.path.isdir(f"{OUT}/{n}"):
        shutil.make_archive(f"{OUT}/{n}", "zip", f"{OUT}/{n}")
print("Download the lora *.zip and result_*.json from the Output tab.")


Download the lora *.zip and result_*.json from the Output tab.
